Data Acquisition & Preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import ast
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Log_Anamoly_Detection/balanced_log_dataset.csv")

def parse_event_sequence(x):
    if pd.isna(x):
        return []
    x = x.strip().strip("[]")
    return [e.strip() for e in x.split(",") if e.strip()]

df['Features'] = df['Features'].apply(parse_event_sequence)

df['Label'] = df['Label'].map({'Success': 0, 'Fail': 1})
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df['Latency_norm'] = scaler.fit_transform(df[['Latency']])


print(df['Features'].head())
print(df['Label'].value_counts())


0    [E5, E5, E5, E22, E11, E9, E11, E9, E11, E9, E...
1    [E22, E5, E5, E5, E26, E26, E11, E9, E11, E9, ...
2    [E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...
3    [E5, E5, E22, E5, E11, E9, E11, E9, E26, E26, ...
4    [E5, E5, E22, E5, E9, E11, E9, E11, E9, E26, E...
Name: Features, dtype: object
Label
1    16838
0    16838
Name: count, dtype: int64


In [ ]:
all_events = set(e for seq in df['Features'] for e in seq)

event2idx = {event: idx + 1 for idx, event in enumerate(all_events)}
event2idx['PAD'] = 0

vocab_size = len(event2idx)


In [ ]:
MAX_LEN = 50

def encode_sequence(seq):
    seq = [event2idx[e] for e in seq]
    return seq[:MAX_LEN] + [0] * max(0, MAX_LEN - len(seq))

df['encoded'] = df['Features'].apply(encode_sequence)
